Genome Firewall
AI-based antimicrobial resistance prediction for Escherichia coli.

Dataset: BV-BRC AMR records

Target: Resistant vs Susceptible

Model: XGBoost classifier

In [4]:
# Imports
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

In [5]:
# Load dataset
df = pd.read_csv("BVBRC_genome_amr.csv")

df.head()

/tmp/ipykernel_566/224238966.py:2: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("BVBRC_genome_amr.csv")


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
0,562,562.193047,Escherichia coli ECBSAC210134367,ciprofloxacin,Resistant,NaN,NaN,NaN,NaN,NaN,...,NaN,BVBRC,NaN,NaN,SIR XGBoost Model,20250225.0,"F1 score: 0.98, CI[0.97, 0.98]",Computational Method,NaN,NaN
1,562,562.227303,Escherichia coli PNUSAE191092,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,BVBRC,NaN,NaN,SIR XGBoost Model,20250225.0,"F1 score: 0.98, CI[0.97, 0.98]",Computational Method,NaN,NaN
2,562,562.668750,Escherichia coli strain JS316,ciprofloxacin,NaN,8.0,NaN,8.00,mg/L,NaN,...,NaN,NaN,NaN,NaN,MIC XGBoost Model,202503101.0,"W1 score: 0.95, CI[0.93, 0.97]",Computational Method,NaN,NaN
3,562,562.362490,Escherichia coli strain CP53,ciprofloxacin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,SIR XGBoost Model,202503101.0,"F1 score: 0.98, CI[0.97, 0.98]",Computational Method,NaN,NaN
4,562,562.205012,Escherichia coli PNUSAE009306,ciprofloxacin,NaN,0.25,NaN,0.25,mg/L,NaN,...,NaN,BVBRC,NaN,NaN,MIC XGBoost Model,20250225.0,"W1 score: 0.95, CI[0.93, 0.97]",Computational Method,NaN,NaN


In [6]:
# Data exploration
df.shape

(242735, 21)

In [7]:
# Preprocessing
df["Resistant Phenotype"].value_counts()

,count
Resistant Phenotype,
Susceptible,96702
Resistant,25486
Intermediate,128


In [8]:
df = df[
    df["Resistant Phenotype"].isin(
        ["Resistant","Susceptible"]
    )
]

df["target"] = (
    df["Resistant Phenotype"]
    .map({
        "Susceptible":0,
        "Resistant":1
    })
)

In [9]:
features = [
    "Taxon ID",
    "Genome ID",
    "Measurement Value"
]

In [10]:
# Train/test split
X = df[features]
y = df["target"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
# Model training
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)


model.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [13]:
# Evaluation
pred = model.predict(X_test)

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.82      0.97      0.89     19341
           1       0.67      0.21      0.32      5097

    accuracy                           0.81     24438
   macro avg       0.75      0.59      0.60     24438
weighted avg       0.79      0.81      0.77     24438



In [14]:
prob = model.predict_proba(X_test)[:,1]

roc = roc_auc_score(
    y_test,
    prob
)

print(
    "ROC-AUC:",
    roc
)

ROC-AUC: 0.8309789971152374


In [15]:
confusion_matrix(
    y_test,
    pred
)

array([[18820,   521],
       [ 4043,  1054]])

In [16]:
# Save model
joblib.dump(
    model,
    "genome_firewall_model.pkl"
)

['genome_firewall_model.pkl']

In [17]:
loaded_model = joblib.load(
    "genome_firewall_model.pkl"
)

loaded_model.predict(
    X_test.iloc[:5]
)

array([0, 0, 0, 0, 0])

In [18]:
# Example prediction
sample = X_test.iloc[[0]]

probability = (
    loaded_model
    .predict_proba(sample)[0][1]
)


if probability > 0.7:
    result = "Likely Resistant"
elif probability < 0.3:
    result = "Likely Susceptible"
else:
    result = "No-call"


print("Prediction:", result)
print(
    "Confidence:",
    round(max(probability,1-probability)*100,2),
    "%"
)

Prediction: No-call
Confidence: 56.19 %
